# Historical unsupported wall-data experiments

These opt-in research cells retain the original delta=0 plus third-derivative
wall-data experiments. The common physical solver deliberately rejects that
boundary combination. The canonical legacy collocation path does not make it
a supported mathematical formulation. Original parameter values, sweeps, and
plotting grids are preserved; inspect convergence before interpreting results.
No cell in this notebook is run by the bounded CI solve tests.


In [ ]:
# Opt-in historical experiment; these boundary rows are outside the public flux problem.
from matplotlib import pyplot as plt
import numpy as np
from overscreening_breakdown.collocation.spectral1d import solve_nonlinear_problem
from overscreening_breakdown.problems.bsk import overscreening_breakdown as LegacyBSK
%matplotlib inline

# Values inherited at the original execution notebook tail are explicit here.
N = 100
L = 6
tolerance = 1.0e-10
use_method = "newton"
initializer = "fsolve"


### Bellow some additional tests are provided.

Feel free to test your parameters as well.

In [ ]:
#set up problem.
#If a parameter is not given, then a default value is used instead. See the source file of "overscreening_breakdown" for more details

sigma = 20.0
mu = 0.0
gamma = 0.1
delta = 0.0
u0 = 0.1
u0xxx = 0

def some_func(x):
    return u0*np.exp(-x**4)

problem = LegacyBSK()  # Explicit historical unsupported wall-data experiment.
problem.set_parameters(sigma = sigma, mu = mu, gamma = gamma, delta = delta, u0 = u0, u0xxx=u0xxx, initial_guess_solution=some_func)

In [ ]:
# Historical unsupported boundary experiment: delta=0 with a third-derivative wall condition.
# Keep the explicit canonical legacy collocation API for this section.
solver = solve_nonlinear_problem(N=N, domain=[0,np.inf], tolerance=tolerance,
                                 use_method=use_method, initializer=initializer, verbosity=2)
solver.set_mapping_parameter(L)
solver.set_problem(problem)
c = solver.solve_problem()
#see the solution
t = np.arange(0, np.pi, 0.01)
sol = solver.obtain_solution_in_basis(c, t)
plt.title(r"solution on the mapped segment [0,$\pi$]")
plt.plot(t, sol)
plt.show()
x = np.arange(0.0, 40.0, 0.1)
sol = solver.obtain_solution_in_domain(c, x)
plt.title(r"solution on the part of the $\mathbb{R}^+$")
plt.plot(x, sol)
plt.show()

In [ ]:
# we perfrom a set of iterations to update a better initial guess from the smaller problem size
def some_func(x):
    return u0*np.exp(-x**4)

L = 1
tolerance = 1.0e-10
N = 60
sigma = 20.0
mu = 0.0
gamma = 1.0
delta = 0.0
u0 = 10.0
u0xxx = 0
u0_c = 0.1
c1 = None
alpha = 1.0

def some_func(x):
    return 2*np.log((1.0+np.tanh(u0/4.0)*np.exp(-x))/(1.0-np.tanh(u0/4.0)*np.exp(-x)))

for it in range(1,1000):
    print("running on gamma = ", gamma)
    solver = solve_nonlinear_problem(N=N, domain=[0,np.inf], tolerance=tolerance, use_method=use_method, total_iterations = 100, use_globalization = False, verbosity=2)
    solver.set_mapping_parameter(L)
    problem = LegacyBSK()  # Explicit historical unsupported wall-data experiment.
    problem.set_parameters(sigma = sigma, mu = mu, gamma = gamma, delta = delta, u0 = u0, u0xxx=u0xxx, initial_guess_solution=some_func)
    solver.set_problem(problem)
    c = solver.solve_problem(c1)
    if solver.is_converged():
        c1 = np.copy(c)
        gamma_old = gamma
        gamma = gamma*0.5
        gamma_found = gamma;
        alpha = 1.0
    else:
        alpha = alpha*0.5
        gamma = (1-alpha)*gamma_old + alpha*gamma_found
    if gamma == 0.0:
        break
    if gamma<1.0e-12:
        gamma = 0.0

In [ ]:
# we apply homotopy when solution fails to converge for some bad RHS
def some_func(x):
    return u0*np.exp(-x**2)
L = 1
tolerance = 1.0e-10
N = 100
sigma = 20.0
mu = 0.0
gamma = 0.0

delta = 0.0
u0 = 2.0
u0xxx = 0
def some_func(x):
    return 2*np.log((1.0+np.tanh(u0/4.0)*np.exp(-x))/(1.0-np.tanh(u0/4.0)*np.exp(-x)))

solver = solve_nonlinear_problem(N=N, domain=[0,np.inf], tolerance=tolerance, use_method=use_method, total_iterations = 10, use_globalization = True, verbosity=2)
solver.set_mapping_parameter(L)
problem = LegacyBSK()  # Explicit historical unsupported wall-data experiment.
problem.set_parameters(sigma = sigma, mu = mu, gamma = gamma, delta = delta, u0 = u0, u0xxx=u0xxx, initial_guess_solution=some_func)
solver.set_problem(problem)
c = solver.solve_problem()

In [ ]:
#see the solution
t = np.arange(0, np.pi, 0.01)
sol = solver.obtain_solution_in_basis(c, t)
plt.title(r"solution on the mapped segment [0,$\pi$]")
plt.plot(t, sol)
plt.show()
x = np.arange(0.0, 5.0, 0.01)
sol = solver.obtain_solution_in_domain(c, x)
plt.title(r"solution on the part of the $\mathbb{R}^+$")
plt.plot(x, sol)
plt.show()